# Bedrock Narrative Reports

Uses Amazon Bedrock (Claude) to generate natural-language scouting narratives for the top players based on their AWI and PQI scores.

**Prerequisites:** `results/awi_full.csv` and `results/pqi_full.csv` must exist (run `run_awi_pipeline` and `run_pqi_pipeline` first).

**Output:** `results/narratives.csv` , one row per player with a generated narrative string.

In [1]:
import os, sys
from pathlib import Path

# Find project root (pyproject.toml marker) and chdir to notebooks/
# so ../results/ and ../figures/ resolve correctly from any launch CWD.
_root = next(
    (p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()),
    None,
)
if _root is None:
    raise RuntimeError("Cannot locate project root  -  pyproject.toml not found.")
os.chdir(_root / "notebooks")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


## Step 1. Initialise Bedrock Client

Creates an authenticated Amazon Bedrock client for the `eu-central-1` region.
Requires a valid AWS session (SSO or environment credentials).

In [2]:
from src.bedrock_client import create_bedrock_client, generate_player_narrative
import logging

client = create_bedrock_client(region="eu-central-1")
print("Bedrock client created successfully")

Bedrock client created successfully


## Step 2. Load AWI and PQI Results

Reads the pre-computed AWI (Awareness Index) and PQI (Pressure Quality Index) CSVs.
These are the inputs to the narrative generation step.

In [3]:
import pandas as pd

awi_df = pd.read_csv("../results/awi_full.csv")
pqi_df = pd.read_csv("../results/pqi_full.csv")
print(f"AWI rows: {len(awi_df)}, PQI rows: {len(pqi_df)}")

AWI rows: 400, PQI rows: 400


## Step 3. Generate Narratives via Bedrock

Generates narratives for the **top 5 players by AWI per match per phase** (= up to 50 narratives across 5 matches × 2 halves). Only players with `awi_per_minute > 0` are included. Each narrative summarises scanning behaviour, pressure quality, and positional context.

In [4]:
merge_keys = ["jersey", "team", "match_id", "phase_label"]
combined = awi_df.merge(pqi_df, on=merge_keys, how="inner", suffixes=("", "_pqi"))
combined["_rank_global"] = combined["awi_per_minute"].rank(ascending=False, method="first").astype(int)
position_avgs = combined.groupby("position")["awi_per_minute"].mean()
total_players = len(combined)

# Top 5 per match per phase, non-zero AWI only
top = (
    combined[combined["awi_per_minute"] > 0]
    .groupby(["match_id", "phase_label"], group_keys=False)
    .apply(lambda g: g.nlargest(5, "awi_per_minute"))
    .reset_index(drop=True)
)
print(f"Generating {len(top)} narratives across {top['match_id'].nunique()} matches")

logger = logging.getLogger(__name__)
records = []

for _, row in top.iterrows():
    try:
        player_row = {
            "name":           row.get("name", f"Player {row['jersey']}"),
            "position":       row.get("position", "Unknown"),
            "match_id":       row["match_id"],
            "phase_label":    row["phase_label"],
            "awi_per_minute": row["awi_per_minute"],
        }
        awi_context = {
            "league_rank":  int(row["_rank_global"]),
            "total_players": total_players,
            "position_avg": position_avgs.get(row.get("position", "Unknown"), 0.0),
        }
        pqi_context = {
            "mean_pqi":         row.get("mean_pqi", 0.0),
            "orientation_mean": row.get("orientation_mean", 0.0),
            "stance_mean":      row.get("stance_mean", 0.0),
            "proximity_mean":   row.get("proximity_mean", 0.0),
        }
        match_context = {"match_label": row["match_id"], "opponent": ""}

        narrative = generate_player_narrative(
            client, player_row, awi_context, pqi_context, match_context
        )
        records.append({
            "jersey":      row["jersey"],
            "team":        row["team"],
            "match_id":    row["match_id"],
            "phase_label": row["phase_label"],
            "narrative":   narrative,
        })
        print(f"  ✓ {player_row['name']} | {row['match_id']} {row['phase_label']} | AWI {row['awi_per_minute']:.1f}")
    except Exception as exc:
        logger.warning(
            "Failed narrative for jersey=%s match=%s phase=%s: %s",
            row.get("jersey"), row.get("match_id"), row.get("phase_label"), exc,
        )

narratives_df = pd.DataFrame(records, columns=["jersey", "team", "match_id", "phase_label", "narrative"])
print(f"\nGenerated {len(narratives_df)} narratives")
narratives_df.head()

Generating 50 narratives across 5 matches


/var/folders/d8/jdj9dp996ps5f3y5m79s3j3m0000gn/T/ipykernel_28042/3242389722.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.nlargest(5, "awi_per_minute"))


  ✓ Tiago Barreiros de Melo Tomás | BVB-VFB 1st half | AWI 17.4
  ✓ Jobe Samuel Patrick Bellingham | BVB-VFB 1st half | AWI 16.6
  ✓ Maximilian Mittelstädt | BVB-VFB 1st half | AWI 16.5
  ✓ Felix Kalu Nmecha | BVB-VFB 1st half | AWI 14.1
  ✓ Deniz Undav | BVB-VFB 1st half | AWI 13.5
  ✓ Tiago Barreiros de Melo Tomás | BVB-VFB 2nd half | AWI 17.4
  ✓ Maximilian Mittelstädt | BVB-VFB 2nd half | AWI 14.0
  ✓ Deniz Undav | BVB-VFB 2nd half | AWI 12.6
  ✓ Julian Ryerson | BVB-VFB 2nd half | AWI 12.0
  ✓ Josha Mamadou Karaboue Vagnoman | BVB-VFB 2nd half | AWI 11.8
  ✓ Joshua Walter Kimmich | FCB-HSV 1st half | AWI 21.8
  ✓ Fábio Daniel Ferreira Vieira | FCB-HSV 1st half | AWI 21.3
  ✓ Nicolai Remberg | FCB-HSV 1st half | AWI 20.5
  ✓ Serge David Gnabry | FCB-HSV 1st half | AWI 20.4
  ✓ Nicolas Capaldo Taboas | FCB-HSV 1st half | AWI 19.6
  ✓ Joshua Walter Kimmich | FCB-HSV 2nd half | AWI 21.1
  ✓ Rayan Philippe | FCB-HSV 2nd half | AWI 20.0
  ✓ Nicolai Remberg | FCB-HSV 2nd half | AWI 18.7


,jersey,team,match_id,phase_label,narrative
0,8,1,BVB-VFB,1st half,"With an Awareness Index of 17.4 scans/min, Tom..."
1,7,0,BVB-VFB,1st half,16.6 scans/min show Jobe Bellingham scans the ...
2,7,1,BVB-VFB,1st half,16.5 scans/min indicate Maximilian Mittelstädt...
3,8,0,BVB-VFB,1st half,With an Awareness Index of 14.1 scans per minu...
4,26,1,BVB-VFB,1st half,"At 13.5 scans per minute, Deniz Undav's Awaren..."


## Step 4. Save and Preview

Persists the narratives to `results/narratives.csv` and prints the top player's narrative as a quick sanity check.

In [5]:
narratives_df.to_csv("../results/narratives.csv", index=False)
print("Saved to ../results/narratives.csv")
print("\n--- Top Player Narrative ---")
if len(narratives_df) > 0:
    print(narratives_df.iloc[0]["narrative"])

Saved to ../results/narratives.csv

--- Top Player Narrative ---
With an Awareness Index of 17.4 scans/min, Tomás's scanning activity is notably higher than his position average of 8.1 scans/min, reflecting his proactive engagement in defensive duties during the first half. His stance score, at 39.3/100, is the weakest component of his Pressure Quality Index, as he often adopts a less effective posture that limits his pressing efficacy. 

To improve his overall pressing, a club should focus on refining his stance mechanics in training, ensuring better body positioning to enhance pressure effectiveness.
